# SpurBreast Colab execution
This notebook runs the reviewed source code from persistent Google Drive. It screens validation-only configurations first, locks the winner, then runs the three final seeds. Test evaluation remains disabled until the lock is committed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/alixdarzi/spurbreast-reproduction.git'
PROJECT = Path('/content/drive/MyDrive/spurbreast-reproduction')
if not PROJECT.exists():
    assert REPO_URL.startswith('https://github.com/alixdarzi/'), 'Unexpected repository URL'
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
status = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True, check=True)
assert not status.stdout, 'Drive repository has uncommitted tracked changes; inspect before updating'
subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)
print(PROJECT)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', '.'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
gpu = torch.cuda.get_device_properties(0)
print(torch.__version__, gpu.name, round(gpu.total_memory / 2**30, 1), 'GiB')
assert gpu.total_memory >= 10 * 2**30, 'Do not silently reduce batch size 32'


In [ ]:
# Safe to rerun: downloads are checksum-verified and preparation is deterministic.
subprocess.run([sys.executable, 'scripts/download_data.py', '--config', 'configs/reproduction.yaml'], check=True)
subprocess.run([sys.executable, 'scripts/prepare_data.py', '--config', 'configs/reproduction.yaml', '--verify-images', '--hash-images'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Validation-only screening
Run one ID per session. Start with H1, H2, H3, and H4. The wrapper resumes the newest matching checkpoint or skips a completed run.

In [ ]:
RUN_ID = 'H1'  # change to H2, H3, H4 in later sessions
config = f'configs/sensitivity_runs/{RUN_ID}.yaml'
subprocess.run([sys.executable, 'scripts/run_or_resume.py', '--config', config, '--device', 'cuda'], check=True)


In [ ]:
# Reports missing primary/fallback/normalization runs using validation summaries only.
subprocess.run([sys.executable, 'scripts/select_sensitivity.py'], check=True)
# Run the requested normalized ID, then rerun this cell.
# Only when status is ready_to_lock:
# subprocess.run([sys.executable, 'scripts/select_sensitivity.py', '--write-lock'], check=True)


## Locked seeds and final evaluation
Commit and push `configs/locked/`, `sensitivity_selection.json`, and the registry before running these cells. Keep final test access false until all three seed checkpoints are complete.

In [ ]:
LOCKED_SEED = 2025  # run 2025, 2026, and 2027, one per session
locked_config = f'configs/locked/seed{LOCKED_SEED}.yaml'
subprocess.run([sys.executable, 'scripts/run_or_resume.py', '--config', locked_config, '--device', 'cuda'], check=True)


In [ ]:
ALLOW_FINAL_TEST = False
assert ALLOW_FINAL_TEST, 'Set true only after lock commit and all three seeds complete'
for seed in (2025, 2026, 2027):
    config = f'configs/locked/seed{seed}.yaml'
    run_dirs = sorted(Path('checkpoints').glob(f'locked_table2_field_strength_resnet50-seed{seed}-*'))
    assert run_dirs, f'Missing seed {seed}'
    checkpoint = run_dirs[-1] / 'best.pt'
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--config', config, '--checkpoint', str(checkpoint), '--splits', 'training', 'validation', 'test', '--allow-test', '--device', 'cuda'], check=True)


## De-identified final tables and figures
Run this CPU-only cell after all three locked evaluations complete. It publishes only aggregate metrics and calibration bins; patient identifiers and per-slice predictions remain in the ignored `results/` directory.

In [ ]:
subprocess.run([sys.executable, 'scripts/summarize_results.py'], check=True)
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('reports/final_results/table2_comparison.csv').round(4))
for figure in ('table2_comparison.png', 'field_strength_shortcut.png', 'test_calibration.png'):
    display(Image(filename=f'reports/final_results/{figure}'))
